# MAP-SAAS と加法 GP モデル

この Notebook では、高次元問題向けの3つの exact GP 系モデル `AdditiveMapSaasSingleTaskGP`、`EnsembleMapSaasSingleTaskGP`、`OrthogonalAdditiveGP` を同じ合成データで比較します。

## このNotebookで学ぶこと

- MAP-SAAS、Ensemble MAP-SAAS、Orthogonal Additive GPの**仮定の違い**
- 共通データでの学習と、予測平均・不確実性の比較
- `qLogExpectedImprovement` による候補提案と実測値の確認
- Mixed版でのカテゴリ探索と、ALEBOとの違い

前の [07_saas_gp.ipynb](07_saas_gp.ipynb) はNUTSによるFully Bayesian SAASを扱いました。本Notebookは、**MAP推定や加法構造を利用する別の選択肢**を比較します。

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import (
    AdditiveMapSaasSingleTaskGP,
    EnsembleMapSaasSingleTaskGP,
    OrthogonalAdditiveGP,
)

torch.manual_seed(0)
dtype = torch.double

### 3モデルの違い

**MAP-SAAS**は、多くの次元の影響が小さいというSAASの疎性事前分布を、MCMCではなく最大事後確率（MAP）推定で扱います。**Ensemble MAP-SAAS**は異なる設定のMAP推定を組み合わせ、単一の推定への依存を緩和します。ただし、完全なMCMC事後分布と同等ではありません。

**Orthogonal Additive GP**は、関数を $f(x)\approx\sum_j f_j(x_j)$ のような低次の成分に分解できるという**関数構造の仮定**を利用します。疎性と加法性は別の性質であり、モデル選択では両方を区別してください。

## 1. 高次元の合成関数

10次元の入力を用意しますが、目的関数に直接効くのは一部の次元だけにしています。高次元かつ疎な関連性を持つ状況を簡略化した例です。

In [ ]:
n, d = 36, 10
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (
        torch.sin(2 * torch.pi * X[..., 0])
        + 0.7 * (X[..., 3] - 0.5) ** 2
        - 0.5 * X[..., 6]
    ).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

### 合成関数の構造

10次元のうち実際に効くのは `x0`, `x3`, `x6` の3次元です。また、真の関数は各変数の寄与を足した形なので**加法構造**も持ちます。このため、今回の例は疎性と加法性の双方を仮定するモデルに有利な設定です。

モデル比較の結果を、すべての高次元問題に一般化することはできません。

## 2. Additive MAP-SAAS

SAAS の疎な関連性の考え方を MAP 推定で扱う加法モデルです。Fully Bayesian SAAS の NUTS より軽量な選択肢として利用できます。

In [ ]:
additive_saas = AdditiveMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(additive_saas.raw_train_X.shape, additive_saas.raw_train_Y.shape)
print('supports_mll =', additive_saas.supports_mll)
additive_saas.fit()

### `num_taus`とMAP学習

`num_taus=3` はMAP-SAASで扱う事前分布のスケール設定に関係します。学習では `fit()` を使用します。内部では `make_mll()` で周辺対数尤度を作成し、BoTorchの `fit_gpytorch_mll()` を呼び出します。Fully Bayesian SAASのようなNUTSサンプリングは行いません。

ここでの値は説明用であり、普遍的な推奨値ではありません。

## 3. Ensemble MAP-SAAS

複数の MAP-SAAS 設定をアンサンブルとして扱うモデルです。Fully Bayesian 推論より計算を抑えつつ、単一の MAP 推定への依存を緩和したい場合に候補になります。

In [ ]:
ensemble_saas = EnsembleMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(ensemble_saas.raw_train_X.shape, ensemble_saas.raw_train_Y.shape)
ensemble_saas.fit()

### アンサンブルの意味

複数のMAP-SAAS構成を組み合わせることで、単一のハイパーパラメータ推定に依存する度合いを下げることを狙います。ただし、アンサンブルが必ず精度を改善するとは限らず、学習・予測のコストも増える場合があります。

## 4. Orthogonal Additive GP

目的関数に一次または低次の加法構造があるという仮定を明示的に利用するモデルです。ここでは `second_order=False` とし、一次の加法構造を使います。

In [ ]:
orthogonal = OrthogonalAdditiveGP(train_X, train_Y, second_order=False)
print(orthogonal.raw_train_X.shape, orthogonal.raw_train_Y.shape)
fit_gpytorch_mll(orthogonal.make_mll())

### 一次加法モデルと相互作用

`second_order=False` は一次成分を中心としたモデル化です。例えば $x_0x_3$ のように**2変数の相互作用**が重要な場合、一次の加法構造だけでは表現しきれない可能性があります。

加法構造が妥当かどうかは、データとドメイン知識に基づいて判断してください。

## 5. 同じ断面で posterior mean を比較

他の入力を 0.5 に固定し、`x0` だけを変化させた断面で各モデルの posterior mean を確認します。

In [ ]:
grid = torch.linspace(0, 1, 80, dtype=dtype)
X_test = torch.full((80, d), 0.5, dtype=dtype)
X_test[:, 0] = grid
models = {
    'additive MAP-SAAS': additive_saas,
    'ensemble MAP-SAAS': ensemble_saas,
    'orthogonal additive': orthogonal,
}
for name, model in models.items():
    with torch.no_grad():
        mean = model.posterior(X_test).mean.squeeze(-1)
    print(name, 'mean range =', (float(mean.min()), float(mean.max())))

### 追加実装例：予測平均と不確実性を重ねて比較

既存コードは各モデルの予測平均の範囲だけを出力します。次の図では `x0` だけを変え、残りの入力を0.5に固定した**1次元スライス**で、真の関数・予測平均・モデル上の約95%区間を比較します。

この図は10次元空間全体の予測性能や、未知データへの汎化精度を保証するものではありません。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(models), figsize=(14, 4), sharey=True)
for ax, (name, surrogate) in zip(axes, models.items()):
    surrogate.eval()
    with torch.no_grad():
        posterior = surrogate.posterior(X_test)
        component_mean = posterior.mean.squeeze(-1)
        component_variance = posterior.variance.squeeze(-1)
        if component_mean.ndim == 2:
            # Ensemble axis: total variance = within + between components.
            prediction = component_mean.mean(dim=0)
            predictive_variance = (
                component_variance.mean(dim=0)
                + component_mean.var(dim=0, unbiased=False)
            )
        else:
            prediction = component_mean
            predictive_variance = component_variance
        uncertainty = predictive_variance.clamp_min(0).sqrt()
    ax.plot(grid, f(X_test).squeeze(-1), "--", label="true function")
    ax.plot(grid, prediction, label="posterior mean")
    ax.fill_between(
        grid,
        prediction - 1.96 * uncertainty,
        prediction + 1.96 * uncertainty,
        alpha=0.2,
        label="approx. 95% latent interval",
    )
    ax.set_title(name)
    ax.set_xlabel("x0")
    ax.legend(fontsize=8)
axes[0].set_ylabel("objective")
plt.tight_layout()
plt.show()

## 6. モデル選択の目安

- **Additive MAP-SAAS**: 高次元で疎な関連性と加法構造の両方が妥当と考えられる場合に向きます。
- **Ensemble MAP-SAAS**: 複数の MAP-SAAS 設定をまとめ、full NUTS より実用的な計算量にしたい場合の候補です。
- **OrthogonalAdditiveGP**: 一次または低次の加法分解そのものが妥当なモデリング仮定である場合に向きます。
- Fully Bayesian SAAS と異なり、これらは exact GP の `make_mll()` による学習フローを利用できます。

## Candidate generation for each additive surrogate

各モデルを同じqLogEI・探索区間で最適化し、提案点と目的関数評価を比較します。

### EIと候補の比較

`qLogExpectedImprovement` は現在の最良観測値を超える**改善の期待値**に基づいて次の候補を提案します。各モデルで同じ `bounds`、`q=1`、最適化設定を使うことで、サロゲートを変えたときの提案の違いを観察できます。

候補の良し悪しは獲得関数値の大小だけでは比較できません。モデルごとに予測分布が異なるためです。合成例では `f(candidate)` による実際の目的値を比較できます。

In [ ]:
bounds = torch.stack([torch.zeros(d, dtype=dtype), torch.ones(d, dtype=dtype)])
for name, surrogate in models.items():
    surrogate.eval()
    acquisition = qLogExpectedImprovement(model=surrogate, best_f=train_Y.max())
    candidate, acquisition_value = optimize_acqf(
        acq_function=acquisition,
        bounds=bounds,
        q=1,
        num_restarts=3,
        raw_samples=32,
    )
    candidate_Y = f(candidate)
    assert candidate.shape == (1, d)
    assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
    print(name, "candidate:", candidate)
    print("acquisition:", acquisition_value, "objective:", candidate_Y)


## Mixed MAP-SAAS models: acquisition-based candidate proposals

カテゴリ列を末尾に保持した入力を使います。カテゴリ値を固定した各探索領域で `optimize_acqf` を実行し、獲得関数値が最大の候補を選択します。


### Mixed版のカテゴリは設計変数

入力の末尾に0または1のカテゴリを追加し、`cat_dims=[d]` で指定します。`fixed_features={d: category}` を使ってカテゴリごとに連続変数を最適化し、獲得関数値が高い候補を採用します。

カテゴリは連続値として補間しません。これはMulti-task GPのtask IDとは異なり、**探索して選択する設計変数**です。

In [ ]:
from robotorchan.models import (
    MixedAdditiveMapSaasSingleTaskGP,
    MixedEnsembleMapSaasSingleTaskGP,
)

mixed_train_X = torch.cat(
    [train_X, torch.arange(n).remainder(2).to(dtype).unsqueeze(-1)],
    dim=-1,
)
mixed_train_Y = f(train_X) + 0.2 * mixed_train_X[:, -1:]
mixed_models = {
    "mixed additive MAP-SAAS": MixedAdditiveMapSaasSingleTaskGP(
        mixed_train_X, mixed_train_Y, cat_dims=[d], num_taus=3
    ),
    "mixed ensemble MAP-SAAS": MixedEnsembleMapSaasSingleTaskGP(
        mixed_train_X, mixed_train_Y, cat_dims=[d], num_taus=3
    ),
}
for name, surrogate in mixed_models.items():
    surrogate.fit()
    surrogate.eval()
    acquisition = qLogExpectedImprovement(
        model=surrogate, best_f=mixed_train_Y.max()
    )
    candidates = []
    for category in (0.0, 1.0):
        fixed_features = {d: category}
        candidate, value = optimize_acqf(
            acq_function=acquisition,
            bounds=torch.stack(
                [torch.zeros(d + 1, dtype=dtype),
                 torch.ones(d + 1, dtype=dtype)]
            ),
            q=1,
            num_restarts=2,
            raw_samples=16,
            fixed_features=fixed_features,
        )
        candidates.append((value.item(), candidate))
    _, candidate = max(candidates, key=lambda result: result[0])
    candidate_Y = f(candidate[:, :d]) + 0.2 * candidate[:, d:]
    assert candidate.shape == (1, d + 1)
    assert candidate[0, d].item() in (0.0, 1.0)
    print(name, "candidate:", candidate, "objective:", candidate_Y)


## ALEBOとMixed Orthogonal Additive GP

ALEBOは既知の観測分散を指定し、Mixed OAKはカテゴリを内部でone-hot符号化します。両モデルとも `qLogExpectedImprovement` と `optimize_acqf` で候補を最適化します。


### ALEBOとMixed OAKの位置付け

**ALEBO**は、低次元の埋め込み構造を仮定して高次元の探索を扱う別のアプローチです。SAASの疎性仮定やOAKの加法性仮定とは異なります。

**Mixed Orthogonal Additive GP**はカテゴリを含む入力で加法構造を使うモデルです。両者を同じNotebookで紹介していますが、モデルの前提は同一ではありません。

In [ ]:
from robotorchan.models import ALEBOGP, MixedOrthogonalAdditiveGP

alebo_model = ALEBOGP(
    train_X, train_Y, train_Yvar=torch.full_like(train_Y, 0.03**2)
)
fit_gpytorch_mll(alebo_model.make_mll())
alebo_model.eval()
alebo_acq = qLogExpectedImprovement(
    model=alebo_model, best_f=train_Y.max()
)
alebo_candidate, albo_value = optimize_acqf(
    acq_function=alebo_acq,
    bounds=bounds,
    q=1,
    num_restarts=2,
    raw_samples=16,
)
assert alebo_candidate.shape == (1, d)
print("ALEBO candidate:", alebo_candidate, "objective:", f(alebo_candidate))

mixed_oak = MixedOrthogonalAdditiveGP(
    mixed_train_X, mixed_train_Y, cat_dims=[d], second_order=False
)
fit_gpytorch_mll(mixed_oak.make_mll())
mixed_oak.eval()
mixed_oak_acq = qLogExpectedImprovement(
    model=mixed_oak, best_f=mixed_train_Y.max()
)
oak_results = []
for category in (0.0, 1.0):
    oak_candidate, oak_value = optimize_acqf(
        acq_function=mixed_oak_acq,
        bounds=torch.stack(
            [torch.zeros(d + 1, dtype=dtype),
             torch.ones(d + 1, dtype=dtype)]
        ),
        q=1,
        num_restarts=2,
        raw_samples=16,
        fixed_features={d: category},
    )
    oak_results.append((oak_value.item(), oak_candidate))
_, oak_candidate = max(oak_results, key=lambda result: result[0])
oak_objective = f(oak_candidate[:, :d]) + 0.2 * oak_candidate[:, d:]
assert oak_candidate.shape == (1, d + 1)
assert oak_candidate[0, d].item() in (0.0, 1.0)
print("Mixed OAK candidate:", oak_candidate, "objective:", oak_objective)


## 7. 追加実装例：各モデルの候補を比較するためのデータ更新

実際のBOでは、選択したモデルが提案した候補を評価したあと、観測データを追加して**同じモデルファミリーを再学習**します。ここでは計算時間を抑えるため、候補の評価とデータ更新を1モデルだけ例示します。

`models` の他のモデルは元データで学習したままです。更新した学習データを渡して新しいモデルを作るだけでは学習は完了しないため、次のコードでは既存の `make_mll()` を使って再学習します。

In [ ]:
update_acq = qLogExpectedImprovement(
    model=additive_saas, best_f=train_Y.max()
)
update_candidate, _ = optimize_acqf(
    acq_function=update_acq,
    bounds=bounds,
    q=1,
    num_restarts=2,
    raw_samples=16,
)
update_Y = f(update_candidate)
updated_X = torch.cat([train_X, update_candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, update_Y.detach()], dim=0)
updated_saas = AdditiveMapSaasSingleTaskGP(
    updated_X, updated_Y, num_taus=3
)
fit_gpytorch_mll(updated_saas.make_mll())
updated_saas.eval()
with torch.no_grad():
    updated_mean = updated_saas.posterior(update_candidate.detach()).mean
assert updated_X.shape == (n + 1, d)
assert updated_Y.shape == (n + 1, 1)
print("evaluated candidate:", update_Y)
print("updated posterior mean:", updated_mean)